# Models

Here I tune my two real models, logistic regression and a decision tree, and compare them with the baselines from notebook 03. Like before, I only use the training set with the same 5 folds. The test set stays untouched.

In [1]:
# Add the project folder to the path so `from src import ...` works when this
# notebook is run from the notebooks folder.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time

import pandas as pd

from src.baselines import SCORING
from src.clean import clean
from src.evaluation import split_by_lifter
from src.features import FEATURES, build_features
from src.models import LR_GRID, TREE_GRID, make_decision_tree, make_logistic_regression, tune
from src.paths import INTERIM, REPORTS

cohort = pd.read_csv(INTERIM / 'audit_cohort.csv.gz', dtype=str, keep_default_na=False)
cleaned, _ = clean(cohort)
table = build_features(cleaned)

# I only keep the training side. The test set isn't used anywhere in this notebook.
train, _ = split_by_lifter(table)
X_train, y_train, groups_train = train[FEATURES], train['made'], train['Name']
print('training rows:', len(X_train))

training rows: 369880


## 1. Logistic regression

In [2]:
start = time.perf_counter()
lr_search = tune(make_logistic_regression(), LR_GRID, X_train, y_train, groups_train)
print(f'took {time.perf_counter() - start:.0f} seconds')
print('best settings:', lr_search.best_params_)

# Mean and std of balanced accuracy across the 5 folds for each C I tried.
lr_table = pd.DataFrame({
    'C': lr_search.cv_results_['param_model__C'],
    'balanced_accuracy_mean': lr_search.cv_results_['mean_test_balanced_accuracy'],
    'balanced_accuracy_std': lr_search.cv_results_['std_test_balanced_accuracy'],
}).round(4)
lr_table

took 94 seconds
best settings: {'model__C': 10}


,C,balanced_accuracy_mean,balanced_accuracy_std
0,0.01,0.6113,0.0015
1,0.10,0.6113,0.0015
2,1.00,0.6113,0.0015
3,10.00,0.6113,0.0015


## 2. Decision tree

In [3]:
start = time.perf_counter()
tree_search = tune(make_decision_tree(), TREE_GRID, X_train, y_train, groups_train)
print(f'took {time.perf_counter() - start:.0f} seconds')
print('best settings:', tree_search.best_params_)

# Mean balanced accuracy across the 5 folds. Rows are max_depth, columns are
# min_samples_leaf.
tree_table = pd.DataFrame({
    'max_depth': tree_search.cv_results_['param_model__max_depth'],
    'min_samples_leaf': tree_search.cv_results_['param_model__min_samples_leaf'],
    'balanced_accuracy_mean': tree_search.cv_results_['mean_test_balanced_accuracy'],
}).pivot(index='max_depth', columns='min_samples_leaf', values='balanced_accuracy_mean')
tree_table.round(4)

took 71 seconds
best settings: {'model__max_depth': 8, 'model__min_samples_leaf': 50}


min_samples_leaf,1,50,200
max_depth,,,
3,0.6030,0.6030,0.6030
5,0.6119,0.6120,0.6120
8,0.6170,0.6174,0.6173
12,0.6122,0.6131,0.6160


## 3. Models next to the baselines

In [4]:
def best_row(search):
    """The fold scores for the setting each search picked, with the same column names
    as baseline_results.csv."""
    i = search.best_index_
    row = {}
    for metric in SCORING:
        row[f'{metric}_mean'] = search.cv_results_[f'mean_test_{metric}'][i]
        row[f'{metric}_std'] = search.cv_results_[f'std_test_{metric}'][i]
    return row

baseline_results = pd.read_csv(REPORTS / 'baseline_results.csv', index_col=0)
models = pd.DataFrame({'logistic regression': best_row(lr_search),
                       'decision tree': best_row(tree_search)}).T
model_results = pd.concat([baseline_results, models])[baseline_results.columns].round(4)
model_results.index.name = 'model'
model_results.to_csv(REPORTS / 'model_results.csv')

# The scores I care about most go first.
first = [f'{m}_{s}' for m in ['balanced_accuracy', 'roc_auc', 'accuracy', 'recall_missed']
         for s in ['mean', 'std']]
model_results[first + [c for c in model_results.columns if c not in first]]

,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std,recall_missed_mean,recall_missed_std,precision_made_mean,precision_made_std,recall_made_mean,recall_made_std,f1_made_mean,f1_made_std,precision_missed_mean,precision_missed_std,f1_missed_mean,f1_missed_std
model,,,,,,,,,,,,,,,,,,
always made,0.5000,0.0000,0.5000,0.0000,0.6102,0.0000,0.0000,0.0000,0.6102,0.0000,1.0000,0.0000,0.7579,0.0000,0.0000,0.0000,0.0000,0.0000
second-attempt rule,0.5553,0.0010,0.5553,0.0010,0.6383,0.0008,0.1792,0.0022,0.6399,0.0006,0.9315,0.0008,0.7586,0.0004,0.6256,0.0030,0.2785,0.0029
logistic regression,0.6113,0.0015,0.6562,0.0019,0.6209,0.0016,0.5680,0.0029,0.7035,0.0013,0.6546,0.0026,0.6782,0.0016,0.5123,0.0018,0.5387,0.0019
decision tree,0.6174,0.0016,0.6629,0.0022,0.6213,0.0036,0.6000,0.0122,0.7131,0.0024,0.6349,0.0132,0.6716,0.0066,0.5122,0.0042,0.5525,0.0033


## Model settings and tuning

I tuned both models with a grid search on the same 5 folds as my baselines, so each lifter stays inside one fold, and I picked the setting with the best mean balanced accuracy. For logistic regression I tried C = 0.01, 0.1, 1 and 10. All four came out at 0.6113 balanced accuracy to 4 decimal places, so C made no real difference here. The search kept C = 10, but it was only ahead after the fourth decimal place. For the decision tree I tried a max depth of 3, 5, 8 and 12 with a minimum leaf size of 1, 50 and 200, which is 12 settings. The best was max depth 8 with a minimum leaf size of 50, at 0.6174. Depth 12 scored lower than depth 8 for every leaf size, so a deeper tree didn't help. Both models use `class_weight='balanced'`. 61% of lifts are made and balanced accuracy is my main score, so without it the models would lean towards always guessing "made".

Both tuned models beat the second-attempt rule on balanced accuracy (0.6113 and 0.6174 against 0.5553) and catch far more of the misses (recall on missed lifts of 0.568 and 0.600 against 0.179). Their plain accuracy is lower than the rule's, though (about 0.621 against 0.638).